In [2]:
#1,
import torch
import torch.nn as nn

class HorizontalAttention(nn.Module):
    def __init__(self, input_dim=768):
        super(HorizontalAttention, self).__init__()
        # In our case, we already have one vector per day
        # So this is just a passthrough, but keeping for architecture completeness
        self.attention_weights = nn.Linear(input_dim, 1)
        
    def forward(self, day_embedding):
        # day_embedding shape: (batch_size, 768)
        # In full implementation, would weight multiple posts per day
        # For now, just return as-is since we pre-aggregated
        return day_embedding   
    

In [3]:
#2, Bi-directional GRU 
class SequentialModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, num_layers=2):
        super(SequentialModel, self).__init__()
        
        self.gru = nn.GRU(
            input_size=input_dim,      # 768 (BERT embedding size)
            hidden_size=hidden_dim,     # 128 (you can tune this)
            num_layers=num_layers,      # 2 layers
            batch_first=True,           # Input shape: (batch, seq, features)
            bidirectional=True,         # Bi-directional
            dropout=0.2 if num_layers > 1 else 0
        )
        
    def forward(self, x):
        # x shape: (batch_size, 30, 768)
        # Output: (batch_size, 30, hidden_dim*2) because bidirectional
        # hidden_dim*2 = 256 if hidden_dim=128
        output, hidden = self.gru(x)
        return output  # Shape: (batch_size, 30, 256)

In [4]:
#3, temporal weights
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim=256):  # 256 because bidirectional GRU (128*2)
        super(TemporalAttention, self).__init__()
        
        # Attention scoring mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
    def forward(self, gru_output):
        # gru_output shape: (batch_size, 30, 256)
        
        # Calculate attention scores for each time step
        attention_scores = self.attention(gru_output)  # (batch_size, 30, 1)
        attention_scores = attention_scores.squeeze(-1)  # (batch_size, 30)
        
        # Softmax to get attention weights (sum to 1)
        attention_weights = torch.softmax(attention_scores, dim=1)  # (batch_size, 30)
        
        # Weighted sum of GRU outputs
        # attention_weights: (batch_size, 30) -> (batch_size, 30, 1)
        # gru_output: (batch_size, 30, 256)
        attention_weights = attention_weights.unsqueeze(-1)
        weighted_output = torch.sum(attention_weights * gru_output, dim=1)  # (batch_size, 256)
        
        return weighted_output, attention_weights.squeeze(-1)  # Return weights for visualization

In [5]:
#4, discriminative network
class PredictionNetwork(nn.Module):
    """
    Final classification layers
    """
    def __init__(self, input_dim=256, hidden_dim=128, output_dim=1):
        super(PredictionNetwork, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, output_dim),
            nn.Sigmoid()  # For binary classification (positive/negative surprise)
        )
        
    def forward(self, x):
        # x shape: (batch_size, 256)
        return self.network(x)  # (batch_size, 1)

In [6]:
class HANModel(nn.Module):
    """
    Complete Hierarchical Attention Network
    """
    def __init__(self, 
                 embedding_dim=768,
                 gru_hidden_dim=128,
                 gru_num_layers=2,
                 prediction_hidden_dim=128):
        super(HANModel, self).__init__()
        
        # Components
        self.horizontal_attention = HorizontalAttention(embedding_dim)
        self.sequential_model = SequentialModel(embedding_dim, gru_hidden_dim, gru_num_layers)
        self.temporal_attention = TemporalAttention(gru_hidden_dim * 2)  # *2 for bidirectional
        self.prediction_network = PredictionNetwork(gru_hidden_dim * 2, prediction_hidden_dim)
        
    def forward(self, x):
        # x shape: (batch_size, 30, 768)
        batch_size, seq_len, embedding_dim = x.shape
        
        # Step 1: Horizontal attention (simplified since we pre-aggregated)
        # In full version, would process multiple posts per day
        # For now, x is already aggregated
        
        # Step 2: Sequential modeling (GRU)
        gru_output = self.sequential_model(x)  # (batch_size, 30, 256)
        
        # Step 3: Temporal attention
        attended_output, attention_weights = self.temporal_attention(gru_output)  # (batch_size, 256)
        
        # Step 4: Prediction
        prediction = self.prediction_network(attended_output)  # (batch_size, 1)
        
        return prediction, attention_weights  # Return attention for visualization

In [15]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HANModel(
    embedding_dim=768,
    gru_hidden_dim=128,
    gru_num_layers=2,
    prediction_hidden_dim=128
).to(device)

# Loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# load data
batch_size = 32

X_train = torch.load(r"D:\baobei\passion projects\Pre-earnings-Social-Media-Sentiment-Accuracy\training\datasets\X_train.pt")
y_train = torch.load(r"D:\baobei\passion projects\Pre-earnings-Social-Media-Sentiment-Accuracy\training\datasets\y_train.pt")
X_test = torch.load(r"D:\baobei\passion projects\Pre-earnings-Social-Media-Sentiment-Accuracy\training\datasets\X_test.pt")
y_test = torch.load(r"D:\baobei\passion projects\Pre-earnings-Social-Media-Sentiment-Accuracy\training\datasets\y_test.pt")
X_val = torch.load(r"D:\baobei\passion projects\Pre-earnings-Social-Media-Sentiment-Accuracy\training\datasets\X_val.pt")
y_val = torch.load(r"D:\baobei\passion projects\Pre-earnings-Social-Media-Sentiment-Accuracy\training\datasets\y_val.pt")

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_38740\2859192337.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_train = torch.load(r"D:\baobei\passion projects\P

In [16]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device).unsqueeze(1)  # Shape: (batch_size, 1)
        
        # Forward pass
        predictions, _ = model(batch_X)  # Ignore attention weights during training
        loss = criterion(predictions, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

In [ ]:
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device).unsqueeze(1)
            
            predictions, _ = model(batch_X)
            loss = criterion(predictions, batch_y)
            total_loss += loss.item()
            
            # Calculate accuracy
            predicted_classes = (predictions > 0.5).float()
            correct += (predicted_classes == batch_y).sum().item()
            total += batch_y.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total
    
    return avg_loss, accuracy

In [18]:
num_epochs = 50
best_val_loss = float('inf')
patience = 10
patience_counter = 0

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_model.pt')
        print("  → Model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

: 

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pt'))

# Evaluate
test_loss, test_accuracy = validate(model, test_loader, criterion, device)
print(f"\nTest Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

# Get predictions for further analysis
model.eval()
all_predictions = []
all_labels = []
all_attention_weights = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        predictions, attention_weights = model(batch_X)
        
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch_y.numpy())
        all_attention_weights.extend(attention_weights.cpu().numpy())

# Convert to arrays
import numpy as np
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)
all_attention_weights = np.array(all_attention_weights)

In [ ]:
import matplotlib.pyplot as plt

# 1. Training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')

plt.subplot(1, 2, 2)
plt.plot(history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Validation Accuracy')

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

# 2. Attention weights visualization (for a few examples)
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

for i in range(3):
    axes[i].bar(range(30), all_attention_weights[i])
    axes[i].set_xlabel('Day')
    axes[i].set_ylabel('Attention Weight')
    axes[i].set_title(f'Example {i+1}: Attention Weights Across 30 Days')
    axes[i].axhline(y=1/30, color='r', linestyle='--', label='Uniform attention')
    axes[i].legend()

plt.tight_layout()
plt.savefig('attention_visualization.png')
plt.show()

# 3. ROC curve and metrics
from sklearn.metrics import roc_curve, auc, classification_report

fpr, tpr, _ = roc_curve(all_labels, all_predictions)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig('roc_curve.png')
plt.show()

# Classification report
predicted_classes = (all_predictions > 0.5).astype(int)
print("\nClassification Report:")
print(classification_report(all_labels, predicted_classes, 
                          target_names=['Negative Surprise', 'Positive Surprise']))